# [Step 5 - JSONLoader] Nested data, chosen granularity

**MLCourse - Agentic AI - Module 05: Document Loaders**

> Stage in the capstone: stage 1 INGEST: the capstone reads user-supplied documents with exactly these loaders

### What you'll learn

- how the `jq_schema` string decides WHICH nodes become Documents
- two granularities side by side: whole-file-as-one-doc vs record-per-doc
- `content_key` for picking one field, `metadata_func` for enriching metadata
- a graceful fallback when the optional `jq` package is missing (this venv!)

---

In [1]:
# =====================================================================
# CELL 1 - SHARED SETUP: imports, track discovery, download-once cache
# =====================================================================

# --- Standard library -------------------------------------------------
import json                    # parsing + pretty-printing JSON payloads
import os                      # file-system odds and ends
import urllib.request          # polite HTTP fetching
from pathlib import Path       # object-oriented filesystem paths

# --- Lesson-specific imports ------------------------------------------
from langchain_community.document_loaders import JSONLoader   # official route
from langchain_core.documents import Document                 # the output shape

# ---------------------------------------------------------------------
# TRACK WALKER - resolve 03_agentic_ai by walking upward from cwd.
# ---------------------------------------------------------------------
def _find_track(start: Path) -> Path:
    """Return the absolute path of the 03_agentic_ai track root."""
    for candidate in (start, *start.parents):
        hit = candidate / "03_agentic_ai"
        if hit.is_dir():
            return hit.resolve()
    raise FileNotFoundError(
        f"No directory named 03_agentic_ai found above {start} - "
        "run this notebook from somewhere inside the MLCourse repo."
    )

TRACK = _find_track(Path.cwd())
DATA = TRACK / "data"
DATA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# DOWNLOAD-ONCE HELPERS - fetch once, cache under DATA, reuse forever.
# ---------------------------------------------------------------------
def get_bytes(fname: str, url: str) -> bytes:
    """Return the file's bytes, downloading only on the very first call."""
    target = DATA / fname
    if target.exists() and target.stat().st_size > 0:
        payload = target.read_bytes()
        print(f"[cache] {fname}: {len(payload):,} bytes")
        return payload
    print(f"[fetch] {url}")
    request = urllib.request.Request(url, headers={"User-Agent": "MLCourse/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = response.read()
    target.write_bytes(payload)
    print(f"[saved] {fname}: {len(payload):,} bytes")
    return payload

def get_text(fname: str, url: str, encoding: str = "utf-8-sig") -> str:
    """get_bytes + decode; drops a BOM if present."""
    return get_bytes(fname, url).decode(encoding, errors="replace")

def to_ascii(text: str) -> str:
    """Console-safe printing for arbitrary text."""
    return text.encode("ascii", errors="replace").decode("ascii")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

C:\Users\Thoyajaksha Kashyap\AppData\Local\Temp\ipykernel_40332\25050113.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader   # official route


### 1. Why JSON needs a schema-driven loader

JSON has no single natural reading unit. One file might hold a thousand
records, one record might BE the file, and the valuable text may sit three
levels deep. `JSONLoader` answers with a mini query language:

- `jq_schema` - a jq expression selecting the nodes that become Documents
  (`"."` = everything; `".[]"` = each element of the top-level array);
- `content_key` - within each selected record, WHICH field is the text;
- `metadata_func` - your hook to copy fields into filterable metadata.

Specimen: JSONPlaceholder's 100 fake blog posts, an array of
`{userId, id, title, body}` objects - realistic API-dump shape.

In [2]:
POSTS_URL = "https://jsonplaceholder.typicode.com/posts"
posts_path = DATA / "posts.json"
raw = get_text("posts.json", POSTS_URL)        # download-once into the cache

records = json.loads(raw)                       # peek at the true structure
print(f"top-level type : {type(records).__name__}, {len(records)} records")
print(f"record keys    : {sorted(records[0])}")
print("--- first record ---")
print(to_ascii(json.dumps(records[0], indent=2)))

[cache] posts.json: 27,520 bytes
top-level type : list, 100 records
record keys    : ['body', 'id', 'title', 'userId']
--- first record ---
{
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}


### 2. Dependency probe: is the optional `jq` engine present?

`JSONLoader` compiles your `jq_schema` with the third-party `jq` package,
which is NOT installed in this course environment. Construction itself
raises ImportError there - so we probe once and, if needed, fall back to a
tiny stand-in that mirrors JSONLoader's behaviour EXACTLY for the two
schemas used today. Same lesson either way; nothing skips.

In [3]:
USING_JQ = True
try:
    JSONLoader(str(posts_path), jq_schema=".")      # constructor compiles schema
except ImportError as err:
    USING_JQ = False
    print("jq package not found -> using MiniJsonLoader stand-in.")
    print(f"   ({err})")

if USING_JQ:
    print("jq available -> the official JSONLoader runs throughout.")

class MiniJsonLoader:
    """Zero-dependency stand-in mirroring JSONLoader for '.' and '.[]'.

    Semantics copied from langchain_community's JSONLoader for THIS lesson's
    schemas and settings: selected node(s) become Documents; dict records
    stringify via json.dumps (that is what text_content=False yields);
    content_key indexes one field; metadata_func REPLACES the metadata dict;
    default metadata carries 'source' (resolved path) and 'seq_num' (1-based).
    """

    def __init__(self, file_path, jq_schema=".", content_key=None,
                 metadata_func=None):
        if jq_schema not in (".", ".[]"):
            raise ValueError("MiniJsonLoader understands only '.' and '.[]'")
        self.file_path = Path(file_path).resolve()
        self.jq_schema = jq_schema
        self.content_key = content_key
        self.metadata_func = metadata_func

    def load(self):
        records = json.loads(self.file_path.read_text(encoding="utf-8"))
        items = records if self.jq_schema == ".[]" else [records]
        docs = []
        for seq_num, rec in enumerate(items, start=1):   # 1-based like the original
            if isinstance(rec, dict) and self.content_key is not None:
                page_content = rec[self.content_key]     # KeyError mirrors original
            elif isinstance(rec, (dict, list)):
                page_content = json.dumps(rec)           # same serialization
            else:
                page_content = "" if rec is None else str(rec)
            metadata = {"source": str(self.file_path), "seq_num": seq_num}
            if self.metadata_func is not None:
                metadata = self.metadata_func(rec, metadata)
            docs.append(Document(page_content=page_content, metadata=metadata))
        return docs

def make_loader(jq_schema, content_key=None, metadata_func=None):
    """Pick the official loader when possible, else the stand-in."""
    if USING_JQ:
        # text_content=False: our records are DICTS, and the official loader
        # refuses non-string content unless we allow serialized output.
        return JSONLoader(str(posts_path), jq_schema=jq_schema,
                          content_key=content_key, metadata_func=metadata_func,
                          text_content=False)
    return MiniJsonLoader(posts_path, jq_schema=jq_schema,
                          content_key=content_key, metadata_func=metadata_func)

jq available -> the official JSONLoader runs throughout.


### 3. Granularity A - the whole file as ONE Document

`jq_schema="."` selects the entire top-level structure. Result: exactly one
Document whose `page_content` is the serialized array. Use-case: tiny files
where global context matters more than searchability. Cost: useless for
retrieval at any real size - one embedding cannot represent 100 posts.

In [4]:
whole_docs = make_loader(".").load()
print(f"Documents : {len(whole_docs)}")
print(f"chars in page_content : {len(whole_docs[0].page_content):,}")
print(f"metadata              : {whole_docs[0].metadata}")
print("--- preview (tail of the serialized array) ---")
print(to_ascii(whole_docs[0].page_content[:220]))

Documents : 1
chars in page_content : 25,318
metadata              : {'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\posts.json', 'seq_num': 1}
--- preview (tail of the serialized array) ---
[{"userId": 1, "id": 1, "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit", "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut qu


### 4. Granularity B - one Document per record

`jq_schema=".[]"` walks INTO the array and yields each element separately:
100 posts become 100 Documents, each with `{"source": path, "seq_num": n}`
metadata. This is the granularity retrieval wants for "find posts about X".

Version detail worth knowing: records are DICTS, so we pass
`text_content=False` - otherwise this loader version refuses non-string
content. The dicts then serialize into readable JSON text.

In [5]:
post_docs = make_loader(".[]").load()
print(f"Documents : {len(post_docs)}")
print(f"--- Document[0] ---")
print(f"content  : {to_ascii(post_docs[0].page_content[:150])}")
print(f"metadata : {post_docs[0].metadata}")

Documents : 100
--- Document[0] ---
content  : {"userId": 1, "id": 1, "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit", "body": "quia et suscipit\nsuscipit recu
metadata : {'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\posts.json', 'seq_num': 1}


### 5. Granularity C - `content_key`: one field as the text

Embedding the full record mixes ids and userIds into the vector. With
`content_key="title"` each Document's text is JUST the title - crisp
embeddings for headline search (and short enough to eyeball below).

In [6]:
title_docs = make_loader(".[]", content_key="title").load()
print(f"Documents : {len(title_docs)}")
for i in (0, 1, 42):
    print(f"  [{i:>2}] {to_ascii(title_docs[i].page_content)}")

Documents : 100
  [ 0] sunt aut facere repellat provident occaecati excepturi optio reprehenderit
  [ 1] qui est esse
  [42] eligendi iste nostrum consequuntur adipisci praesentium sit beatae perferendis


### 6. `metadata_func` plus a real manipulation: filter by author

`metadata_func(record, metadata)` receives each selected record and the
default metadata dict; whatever you return becomes final metadata. We copy
`userId`/`id` out of the content zone into filterable keys - then run the
classic manipulation: keep only user 7's posts.

In [7]:
def add_user_id(record: dict, metadata: dict) -> dict:
    """Copy identity fields from the record into filterable metadata."""
    metadata["user_id"] = record.get("userId")   # author facet
    metadata["post_id"] = record.get("id")       # stable citation handle
    return metadata

user_docs = make_loader(".[]", metadata_func=add_user_id).load()

print("sample metadata:", user_docs[0].metadata)

user7 = [doc for doc in user_docs if doc.metadata.get("user_id") == 7]
print(f"\nposts by user 7 : {len(user7)} of {len(user_docs)}")
for doc in user7[:3]:
    body_head = to_ascii(json.loads(doc.page_content)["body"])[:70]
    print(f"  post {doc.metadata.get('post_id')}: {body_head}...")

sample metadata: {'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\posts.json', 'seq_num': 1, 'user_id': 1, 'post_id': 1}

posts by user 7 : 10 of 100
  post 61: ab nemo optio odio
delectus tenetur corporis similique nobis repellend...
  post 62: enim aspernatur illo distinctio quae praesentium
beatae alias amet del...
  post 63: enim adipisci aspernatur nemo
numquam omnis facere dolorem dolor ex qu...


### 7. Pitfalls worth remembering

**Pitfall - granularity is destiny**: whole-file mode made ONE blob; per-post
mode made 100 retrievable units. Wrong here means retrieval never had a
chance - choose `.[]`-style schemas for record collections, always.

**Pitfall - the jq dependency is optional but real**: `JSONLoader` needs
`pip install jq`. This notebook degrades gracefully; deeper nested documents
(schemas beyond `.` / `.[]`) REQUIRE the genuine jq expression engine.

**Pro-tip - metadata_func is your filter-index builder**: anything retrieval
will filter or cite on (ids, dates, categories) belongs in metadata, NOT in
the embedded text.

### Takeaway

**JSONLoader turns a jq query into Documents: `.` = one blob, `.[]` =
record-per-doc, `content_key` narrows the text, `metadata_func` builds the
filterable address labels. Granularity first, fields second.**

### Summary

- Same 27 KB file produced 1 Document (`.`), 100 Documents (`.[]`), and
  100 title-only Documents (`content_key="title"`) - three different
  retrieval universes from one schema string.
- `metadata_func=add_user_id` moved identity into metadata, enabling the
  user-7 filter (10 posts) without touching page_content.
- Missing `jq`? A labeled stand-in kept every cell runnable - and showed
  exactly which behaviour the real package generalizes.